# Step 05B — Single Candidate Cross-Dataset Image Comparison

**Purpose:** Display the *one* perceptual-similarity candidate flagged in Step 05. This is a manual image-overlap check, **not** patient identity verification. No training or GPU, no change to source images or labels.

- Internal: `processed_data/yolo_caries_baseline/images/test/cate2_004.png`
- External: `external_validation/MOPG7_66/images/test/0000837.jpg`

Run the cells. Inspect the two radiographs at full resolution and the side-by-side preview. Record one outcome: `clearly the same radiograph`, `clearly different radiographs`, or `uncertain`. Distinct radiographs do not prove distinct patients.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import display
import hashlib, json
R=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
A=R/'processed_data/yolo_caries_baseline/images/test/cate2_004.png'
B=R/'external_validation/MOPG7_66/images/test/0000837.jpg'
for f in [A,B]:
    assert f.is_file(), f'Missing expected file: {f}'
print('Paths found. SHA-256:')
for f in [A,B]:
    h=hashlib.sha256(f.read_bytes()).hexdigest()
    with Image.open(f) as im: print(f.name, im.size, im.mode, h)

# Preserve originals, display only a scaled copy for inspection.
imgs=[]
for f in [A,B]:
    with Image.open(f) as im:
        imgs.append(ImageOps.exif_transpose(im).convert('L').copy())

CANVAS_W=2200
half=CANVAS_W//2
TARGET_H=750
panel=Image.new('RGB',(CANVAS_W,TARGET_H+78),'#eef1f4')
draw=ImageDraw.Draw(panel)
for i,(f,im) in enumerate(zip([A,B],imgs)):
    fit=ImageOps.contain(im,(half-24,TARGET_H-24))
    x=i*half+(half-fit.width)//2
    y=78+(TARGET_H-fit.height)//2
    panel.paste(Image.merge('RGB',(fit,fit,fit)),(x,y))
    draw.text((i*half+25,25),('INTERNAL  ' if i==0 else 'EXTERNAL  ')+f.name,fill='#1d3248')
OUT=R/'reviewer_revision'/'Step_05B_Visual_Pair_Review'
OUT.mkdir(parents=True,exist_ok=True)
PREVIEW=OUT/'candidate_pair_side_by_side.png'
panel.save(PREVIEW)
print('Saved:',PREVIEW)
display(panel)


## Record the visual conclusion

Review full-resolution originals if the overview is insufficient. Write the outcome in the last cell and keep that text/PNG as audit evidence. No demographic or patient-identity conclusion follows from this screen.

In [ ]:
# AFTER looking at the pair, change ONLY the following two values.
# Choose exactly: 'same_radiograph', 'different_radiographs', or 'uncertain'
MANUAL_RESULT = 'uncertain'
REVIEWER_INITIALS = 'not_recorded'
VALID = {'same_radiograph','different_radiographs','uncertain'}
assert MANUAL_RESULT in VALID
record={'internal_image':A.name,'external_image':B.name,
        'comparison':'manual inspection of radiographic appearance',
        'visual_result':MANUAL_RESULT, 'reviewer_initials':REVIEWER_INITIALS,
        'limitation':'image comparison cannot rule out patient overlap'}
record_path=OUT/'candidate_pair_visual_review.json'
record_path.write_text(json.dumps(record,indent=2),encoding='utf-8')
print(record_path)
print(json.dumps(record,indent=2))
